In [1]:
#DataScienceCourse/
#├── Background/               # Cheatsheets and reference manuals
#├── docs/                     # Exported Markdown notebooks for GitHub Pages
#├── notebooks/
#│   ├── 00_ProjectTemplate.ipynb
#│   └── SX.Y_DESC.ipynb       # Sequential step notebooks
#├── persisted/                # Final serialized joblib artifacts
#│   ├── descriptive/
#│   ├── models/
#│   └── framework/
#├── src/                      # Flat Python package (auto-discovered by setuptools)
#│   ├── __init__.py
#│   ├── paths.py              # OS-agnostic relative path resolution
#│   ├── descriptive_tools.py  # Statistical profiling & ESS metadata helpers
#│   ├── s0_screening.py       # Step 0: Exploratory screening & ESS cataloging
#│   ├── s1_ingestion.py       # Step 1: APIs, scraping & SQL persistence
#│   ├── s2_cleaning.py        # Step 2: Survey sanitization, macro merge & audit flags
#│   ├── s3_eda.py             # Step 3: Exploratory stats persistence
#│   ├── s4_features.py        # Step 4: Deterministic feature engineering
#│   ├── s5_modeling.py        # Step 5: GroupKFold modeling & benchmarking
#│   ├── wrapper.py            # Step 6.1: Global orchestrator & ablation runner
#│   ├── apps_dash/            # Standalone Plotly Dash sandbox
#│   ├── apps_fapi/            # Standalone FastAPI sandbox
#│   └── apps_flask/           # Standalone Flask sandbox
#├── temp/                     # Transitional checkpoints, raw HTML/XML (gitignored)
#├── tests/                    # Pytest data-contract & pipeline tests
#├── pyproject.toml            # Single source of truth for dependencies & CLI
#├── ProjectStatement.md       # Technical specification
#└── Index_Readme.md           # Sequential workflow guide
import sys
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from paths import setup_workspace

PATHS = setup_workspace()


[OK] Structural folder check passed: all required directories exist.
[OK] Workspace anchored at ROOT: /home/josu/Documentos/DataScienceCourse (relative base: '.')


In [3]:
# ==============================================================================
# NOTEBOOK SCOPE & SEQUENTIAL PIPELINE:
#
# [ACTIVE] STEP 0.1 — DuckDB Pushdown Filtering & Initial Sanitization (Layer 1):
#                     Stream raw ESS11 data in memory-safe 5,000-row chunks via
#                     DuckDB using the embedded L0 metadata dictionary
#                     (ess11.duckdb -> l0_valid_cols).
#
# SEQUENTIAL EXECUTION ORDER:
#   Step A) Constants & Configuration:
#           Define unified DuckDB path (ess11.duckdb), table contracts (L0/L1),
#           chunk size (5,000 rows), checkpoint path, null threshold (35%),
#           protected identifiers/targets, and non-response code masks.
#   Step B) Embedded Dictionary Query & Chunked SQL Generation:
#           Load l0_valid_cols directly from DuckDB, map variable-specific
#           non-response codes (preserving 6/66 "Not applicable" routing codes),
#           and build a parameterized SQL query that filters round 11, computes
#           pre-imputation audit signals, sanitizes non-responses to typed NULL,
#           and applies dtype-safe COALESCE harmonization.
#   Step C) Chunked Execution, Column Pruning & Relational Persistence:
#           Iterate over 5,000-row blocks with JSON cursor checkpointing,
#           prune columns exceeding >35% real NULLs, and persist l1_ess11_clean,
#           l1_pruned_cols, and l1_manifest inside ess11.duckdb.
# ==============================================================================
# ==============================================================================
# S0.1- L1 Step A) CONSTANTS:
# ==============================================================================
import json
import re
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
import duckdb
import pandas as pd

TEMP_DIR = PATHS["TABLES"]                  # ./temp
DB_PATH = TEMP_DIR / "ess11.duckdb"         # Unified relational container (L0 + L1)
L0_META_TABLE = "l0_valid_cols"             # Embedded L0 metadata table
L1_DATA_TABLE = "l1_ess11_clean"            # Final cleaned L1 data table
STAGE_PARQUET = TEMP_DIR / "ess11_raw_slice.parquet"
# Adjust to your local path (CSV/Parquet) or Hugging Face URL
# RAW_SOURCE = "https://huggingface.co/datasets/JBG89Spain/ESSsurveys.sav/resolve/main/ESSproject.csv"
RAW_SOURCE = TEMP_DIR / "ESSproject.csv"
LOCAL_RAW_CACHE = TEMP_DIR / "ESSproject_raw.csv"
CHECKPOINT_PATH = TEMP_DIR / "l1_checkpoint.json"

CHUNK_SIZE = 5000       # Adaptive row block size to prevent OOM spikes
NULL_THRESHOLD = 0.35   # Discard columns with > 35% actual NULL (excluding 6/66) after coalescing

# Pure non-response codes (Refusal, Don't know, No answer).
# DOES NOT include 6 / 66 / 666 / 6666 ("Not applicable"), which are preserved intact.
NON_RESPONSE_CODES = {
    7, 8, 9,
    77, 88, 99,
    777, 888, 999,
    7777, 8888, 9999,
    77777, 88888, 99999,
    7777777, 8888888, 9999999,
}

# Variables that must never be recoded or dropped due to null thresholds
PROTECTED_COLS = {
    "name", "essround", "edition", "proddate", "idno", "cntry",
    "dweight", "pspwght", "pweight", "anweight", "prob", "stratum", "psu",
    "lrscale", "happy"
}


# ==============================================================================
# S0.1- L1 Step B) DICTIONARY PARSING UTILITIES AND SQL GENERATION
# ==============================================================================
def ensure_local_source(source: str) -> str:
    """Download remote CSV via 1MB streaming chunks to disk so DuckDB can page without HTTP re-reads."""
    if not str(source).startswith(("http://", "https://")):
        return str(source)
    if LOCAL_RAW_CACHE.exists() and LOCAL_RAW_CACHE.stat().st_size > 10_000_000:
        print(f"[CACHE] Using local raw CSV -> {LOCAL_RAW_CACHE}")
        return str(LOCAL_RAW_CACHE)

    print(f"[DOWNLOAD] Streaming remote CSV to {LOCAL_RAW_CACHE}...")
    with urllib.request.urlopen(source) as resp, open(LOCAL_RAW_CACHE, "wb") as f:
        while chunk := resp.read(1024 * 1024):
            f.write(chunk)
    print(f"[OK] Download complete ({LOCAL_RAW_CACHE.stat().st_size / (1024**3):.2f} GB)")
    return str(LOCAL_RAW_CACHE)


def load_valid_dict_from_db(
    con: duckdb.DuckDBPyConnection,
    table_name: str = L0_META_TABLE,
) -> dict[str, dict]:
    """Query embedded L0 metadata dictionary directly from DuckDB (ess11.duckdb)."""
    tables = {r[0] for r in con.execute("SHOW TABLES").fetchall()}
    if table_name not in tables:
        raise RuntimeError(
            f"[ERROR] Table '{table_name}' not found in {DB_PATH}. Run Step 0.0 (L0) first."
        )

    df_meta = con.execute(f"""
        SELECT
            var_name,
            sector,
            description,
            range_str,
            var_kind,
            min_val,
            max_val,
            coalesce_from_json,
            labels_json
        FROM {table_name}
    """).df()

    flat = {}
    for row in df_meta.itertuples(index=False):
        flat[row.var_name] = {
            "sector": row.sector,
            "description": row.description,
            "range": row.range_str,
            "var_kind": row.var_kind,
            "min_val": row.min_val,
            "max_val": row.max_val,
            "coalesce_from": json.loads(row.coalesce_from_json) if row.coalesce_from_json else [],
            "labels": json.loads(row.labels_json) if row.labels_json else {},
        }
    return flat


def get_non_resp_codes_for_var(var_name: str, meta: dict) -> list[int]:
    """
    Detect which NON_RESPONSE_CODES apply to this variable without deleting
    valid values (such as 7, 8, 9 on 0-10 scales or ages 77/88/99 on agea).
    """
    if var_name in PROTECTED_COLS:
        return []

    range_str = meta.get("range", "")
    labels = meta.get("labels", {})

    # Continuous variables (e.g., agea 15-90, netpay, wkhtot): use only 3+ digit codes
    if "cont" in range_str:
        m = re.search(r"(\d+)\s*-\s*(\d+)", range_str)
        max_v = int(m.group(2)) if m else 90
        return sorted([c for c in NON_RESPONSE_CODES if c > max_v and c >= 777])

    # String variables (e.g., cntry, isco08)
    if "string" in range_str:
        return []

    # Discrete variables: extract valid upper bound
    m = re.search(r"(\d+)\s*-\s*(\d+)", range_str)
    if not m:
        return []
    max_valid = int(m.group(2))

    # Filter reserved codes strictly greater than the valid maximum
    # while preserving 6 / 66 / 666 / 6666 ("Not applicable")
    valid_label_keys = {int(k) for k in labels.keys() if k.isdigit()}
    codes = [
        c for c in NON_RESPONSE_CODES
        if c > max_valid and (not valid_label_keys or c not in valid_label_keys)
    ]
    return sorted(codes)


def build_chunk_sql(
    con: duckdb.DuckDBPyConnection,
    source_expr: str,
    valid_meta: dict[str, dict],
) -> tuple[str, list[str]]:
    """
    Construct a parameterized DuckDB SQL template preserving native column dtypes:
    1. Slice raw rows via {limit} and {offset} projecting only required columns.
    2. Filter WHERE essround = 11 as a strict prerequisite before sanitization/coalescing.
    3. Compute intra-subject audit signals (audit_evasion_ratio, audit_midpoint_rate) BEFORE NULL recoding.
    4. Recode 7/8/9 non-response families -> typed NULL (preserving 6/66 routing codes).
    5. Apply dtype-safe COALESCE harmonization across round variants.
    """
    # Discover physical schema and inferred native types without loading the full file
    schema_df = con.execute(f"DESCRIBE SELECT * FROM {source_expr}").df()
    physical_cols = set(schema_df["column_name"].str.lower())
    physical_map = {c.lower(): c for c in schema_df["column_name"]}
    type_map = {c.lower(): t for c, t in zip(schema_df["column_name"], schema_df["column_type"])}

    # Map raw source variants (e.g., impdiffa, edulvlb) to their parent canonical metadata
    # so non-response codes (7/8/9) are sanitized to typed NULL BEFORE coalescing
    raw_to_meta = {k.lower(): v for k, v in valid_meta.items()}
    for canonical_col, meta in valid_meta.items():
        for src_col in meta.get("coalesce_from", []):
            raw_to_meta.setdefault(src_col.lower(), meta)

    # Collect all required raw columns (canonical + coalesce sources + protected)
    needed_raw = set()
    for col, meta in valid_meta.items():
        if col.lower() in physical_cols:
            needed_raw.add(physical_map[col.lower()])
        for src_col in meta.get("coalesce_from", []):
            if src_col.lower() in physical_cols:
                needed_raw.add(physical_map[src_col.lower()])

    for pcol in PROTECTED_COLS:
        if pcol.lower() in physical_cols:
            needed_raw.add(physical_map[pcol.lower()])

    needed_raw_sorted = sorted(needed_raw)
    raw_select_sql = ", ".join(f'"{c}"' for c in needed_raw_sorted)

    # Prerequisite 1: Round 11 filter applied immediately after slicing
    where_clause = "WHERE TRY_CAST(essround AS INTEGER) = 11" if "essround" in physical_cols else ""

    # Step 2: Recode 7/8/9 -> typed NULL (keeping 6/66 intact) and compute audit metrics
    clean_exprs = []
    evasion_terms = []
    midpoint_terms = []

    for col in needed_raw_sorted:
        c_low = col.lower()
        orig_type = type_map[c_low]
        meta = raw_to_meta.get(c_low, {})
        nr_codes = get_non_resp_codes_for_var(c_low, meta)

        if nr_codes and "string" not in meta.get("range", "") and "VARCHAR" not in orig_type.upper():
            codes_sql = ", ".join(str(x) for x in nr_codes)
            clean_exprs.append(
                f'CASE WHEN TRY_CAST("{col}" AS BIGINT) IN ({codes_sql}) '
                f'THEN CAST(NULL AS {orig_type}) ELSE "{col}" END AS "{col}"'
            )
            # Count evasion on attitudinal/core items (excluding targets to prevent leakage)
            if c_low not in {"lrscale", "happy"} and not c_low.startswith(("rship", "gndr", "yrbrn")):
                evasion_terms.append(
                    f'CASE WHEN TRY_CAST("{col}" AS BIGINT) IN ({codes_sql}) THEN 1.0 ELSE 0.0 END'
                )
        else:
            clean_exprs.append(f'"{col}"')

        # Midpoint response rate (5 on 0-10 scales) to detect neutral-refuge bias
        if meta.get("range") == "0 - 10 discrete" and c_low not in {"lrscale", "happy"}:
            midpoint_terms.append(
                f'CASE WHEN TRY_CAST("{col}" AS BIGINT) = 5 THEN 1.0 ELSE 0.0 END'
            )

    evasion_sql = (
        f"({' + '.join(evasion_terms)}) / {len(evasion_terms)}.0 AS audit_evasion_ratio"
        if evasion_terms else "0.0 AS audit_evasion_ratio"
    )
    midpoint_sql = (
        f"({' + '.join(midpoint_terms)}) / {len(midpoint_terms)}.0 AS audit_midpoint_rate"
        if midpoint_terms else "0.0 AS audit_midpoint_rate"
    )

    # Step 3: Harmonized pair coalescing with uniform dtype casting
    coalesce_exprs = []
    final_cols = []

    for col, meta in valid_meta.items():
        c_low = col.lower()
        sources = [s for s in meta.get("coalesce_from", []) if s.lower() in physical_cols]
        is_string_var = "string" in meta.get("range", "")

        if len(sources) >= 2:
            # Prioritize ESS11 variants ending in 'a'/'b'
            sources_ordered = sorted(sources, key=lambda x: (0 if x.endswith(("a", "b")) else 1, x))
            phys_sources = [physical_map[s.lower()] for s in sources_ordered]

            if (
                is_string_var
                or (
                    any("VARCHAR" in type_map[s.lower()].upper() for s in sources_ordered)
                    and "cont" not in meta.get("range", "")
                    and "discrete" not in meta.get("range", "")
                )
            ):
                coalesce_args = ", ".join(
                    f'NULLIF(TRIM(CAST("{p}" AS VARCHAR)), \'\')' for p in phys_sources
                )
            else:
                target_num_type = (
                    "DOUBLE"
                    if any(
                        t in type_map[s.lower()].upper()
                        for s in sources_ordered
                        for t in ("DOUBLE", "FLOAT", "DECIMAL")
                    )
                    else "BIGINT"
                )
                coalesce_args = ", ".join(f'TRY_CAST("{p}" AS {target_num_type})' for p in phys_sources)

            coalesce_exprs.append(f'COALESCE({coalesce_args}) AS "{col}"')
            final_cols.append(col)
        elif c_low in physical_cols:
            phys = physical_map[c_low]
            coalesce_exprs.append(f'"{phys}" AS "{col}"')
            final_cols.append(col)

    # Append intra-subject audit features computed prior to NULL recoding
    coalesce_exprs.extend(["audit_evasion_ratio", "audit_midpoint_rate"])
    final_cols.extend(["audit_evasion_ratio", "audit_midpoint_rate"])

    sql_template = f"""
    WITH raw_slice AS (
        SELECT {raw_select_sql}
        FROM {source_expr}
        LIMIT {{limit}} OFFSET {{offset}}
    ),
    raw_ess11 AS (
        SELECT * FROM raw_slice
        {where_clause}
    ),
    sanitized AS (
        SELECT
            {', '.join(clean_exprs)},
            {evasion_sql},
            {midpoint_sql}
        FROM raw_ess11
    )
    SELECT
        {', '.join(coalesce_exprs)}
    FROM sanitized
    """
    return sql_template, final_cols

In [4]:
# ==============================================================================
# S0.1- L1 Step C) STREAMING BRIDGE -> SANITIZE & COALESCE -> GLOBAL NULL PRUNING
# ==============================================================================



def run_duckdb_cleaning(reset_checkpoint: bool = False):
    local_path = ensure_local_source(str(RAW_SOURCE))

    with duckdb.connect(str(DB_PATH)) as con:
        con.execute("SET preserve_insertion_order = false;")
        con.execute("SET threads = 1;")
        con.execute("SET memory_limit = '3GB';")
        con.execute(f"SET temp_directory = '{TEMP_DIR / 'duckdb_spill'}';")

        valid_meta = load_valid_dict_from_db(con, L0_META_TABLE)

# ==============================================================================
# PHASE 1: Single-pass CSV -> ESS11 columnar Parquet bridge (all raw variants)
# ==============================================================================
        if reset_checkpoint or not STAGE_PARQUET.exists():
            print("[1/4] Streaming raw CSV -> ESS11 columnar Parquet bridge...")
            schema_df = con.execute(
                f"DESCRIBE SELECT * FROM read_csv_auto('{local_path}', ignore_errors=true)"
            ).df()
            phys_map = {c.lower(): c for c in schema_df["column_name"]}

            needed_raw = sorted({
                phys_map[c.lower()]
                for col, meta in valid_meta.items()
                for c in ([col] + meta.get("coalesce_from", []))
                if c.lower() in phys_map
            } | {phys_map[p.lower()] for p in PROTECTED_COLS if p.lower() in phys_map})

            cols_sql = ", ".join(f'"{c}"' for c in needed_raw)
            con.execute(f"""
                COPY (
                    SELECT {cols_sql}
                    FROM read_csv_auto('{local_path}', ignore_errors=true, sample_size=5000)
                    WHERE TRY_CAST(essround AS INTEGER) = 11
                ) TO '{STAGE_PARQUET}' (FORMAT PARQUET, COMPRESSION ZSTD);
            """)
            print(f"   -> Persisted ESS11 bridge: {STAGE_PARQUET.stat().st_size / (1024**2):.1f} MB")
# ==============================================================================
# PHASE 2: Sanitize 7/8/9 -> NULL FIRST, then COALESCE into canonical cols
# ==============================================================================
        parquet_expr = f"read_parquet('{STAGE_PARQUET}')"
        countries = [
            r[0] for r in con.execute(
                f"SELECT DISTINCT cntry FROM {parquet_expr} WHERE cntry IS NOT NULL ORDER BY cntry"
            ).fetchall()
        ]
        print(f"[2/4] Sanitizing & coalescing {len(countries)} countries into unified stage...")

        clean_sql_base, candidate_cols = build_chunk_sql(con, parquet_expr, valid_meta)
        clean_sql_no_limit = clean_sql_base.replace("LIMIT {limit} OFFSET {offset}", "")

        con.execute("DROP TABLE IF EXISTS ess11_stage;")

        for idx, cntry in enumerate(countries):
            cntry_sql = clean_sql_no_limit.replace(
                "WHERE TRY_CAST(essround AS INTEGER) = 11",
                f"WHERE cntry = '{cntry}'"
            )
            if idx == 0:
                con.execute(f"CREATE TABLE ess11_stage AS {cntry_sql}")
            else:
                con.execute(f"INSERT INTO ess11_stage {cntry_sql}")

        n_rows = con.execute("SELECT COUNT(*) FROM ess11_stage").fetchone()[0]
        print(f"   -> Staged ESS11 rows: {n_rows:,} | Coalesced candidate columns: {len(candidate_cols)}")

# ==============================================================================
# PHASE 3 (FINAL STEP): Global >35% NULL pruning AFTER coalescing
# ==============================================================================
        null_checks = ", ".join(
            f'AVG(CASE WHEN "{c}" IS NULL THEN 1.0 ELSE 0.0 END) AS "{c}"'
            for c in candidate_cols
        )
        null_rates = con.execute(f"SELECT {null_checks} FROM ess11_stage").df().iloc[0].to_dict()

        kept_cols = [
            c for c in candidate_cols
            if c in PROTECTED_COLS or c.startswith("audit_") or null_rates.get(c, 0.0) <= NULL_THRESHOLD
        ]
        dropped_cols = [
            (c, round(null_rates.get(c, 0.0) * 100, 1))
            for c in candidate_cols if c not in kept_cols
        ]

        kept_sql = ", ".join(f'"{c}"' for c in kept_cols)
        country_tables = []

        # Materialize clean country partitions with the final pruned schema
        for cntry in countries:
            tbl_name = f"l1_cntry_{cntry.lower()}"
            con.execute(f"""
                CREATE OR REPLACE TABLE {tbl_name} AS
                SELECT {kept_sql}
                FROM ess11_stage
                WHERE cntry = '{cntry}'
            """)
            country_tables.append(tbl_name)

        unified_view_sql = " UNION ALL ".join(f"SELECT * FROM {t}" for t in country_tables)
        con.execute(f"CREATE OR REPLACE VIEW {L1_DATA_TABLE} AS {unified_view_sql}")
        con.execute("DROP TABLE ess11_stage;")

        # Persist discarded column audit & L1 manifest inside ess11.duckdb
        df_pruned = pd.DataFrame(dropped_cols, columns=["var_name", "null_pct"])
        con.execute("CREATE OR REPLACE TABLE l1_pruned_cols AS SELECT * FROM df_pruned")

        df_l1_manifest = pd.DataFrame([{
            "layer": "L1",
            "ess_round": "ESS11",
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "n_countries": len(country_tables),
            "n_rows": n_rows,
            "n_kept_cols": len(kept_cols),
            "n_dropped_cols": len(dropped_cols),
            "null_threshold": NULL_THRESHOLD,
        }])
        con.execute("CREATE OR REPLACE TABLE l1_manifest AS SELECT * FROM df_l1_manifest")

        print(f"[3/4] Post-coalesce pruning (> {int(NULL_THRESHOLD*100)}% NULL): Kept {len(kept_cols)} | Dropped {len(dropped_cols)}")
        print(f"[4/4] Persisted {len(country_tables)} country tables (l1_cntry_*) + VIEW '{L1_DATA_TABLE}' in {DB_PATH.name}")


if __name__ == "__main__":
    run_duckdb_cleaning(reset_checkpoint=True)

[1/4] Streaming raw CSV -> ESS11 columnar Parquet bridge...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   -> Persisted ESS11 bridge: 2.7 MB
[2/4] Sanitizing & coalescing 30 countries into unified stage...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (24 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   -> Staged ESS11 rows: 50,116 | Coalesced candidate columns: 347
[3/4] Post-coalesce pruning (> 35% NULL): Kept 136 | Dropped 211
[4/4] Persisted 30 country tables (l1_cntry_*) + VIEW 'l1_ess11_clean' in ess11.duckdb


In [5]:
'''with duckdb.connect(str(DB_PATH), read_only=True) as con:
    display(con.execute("SHOW TABLES").df())
    display(con.execute("SELECT * FROM l1_manifest").df())
    display(con.execute("SELECT * FROM l1_pruned_cols ORDER BY null_pct DESC LIMIT 15").df())
    display(con.execute("SELECT cntry, COUNT(*) AS n FROM l1_ess11_clean GROUP BY cntry ORDER BY n DESC").df())'''

,name
0,l0_blacklist
1,l0_manifest
2,l0_valid_cols
3,l1_cntry_at
4,l1_cntry_be
5,l1_cntry_bg
6,l1_cntry_ch
7,l1_cntry_cy
8,l1_cntry_de
9,l1_cntry_ee


,layer,ess_round,created_utc,n_countries,n_rows,n_kept_cols,n_dropped_cols,null_threshold
0,L1,ESS11,2026-11-25T10:41:23.069198+00:00,30,50116,136,211,0.35


,var_name,null_pct
0,accdng,100.0
1,actvgrs,100.0
2,blgetmg,100.0
3,bsmw,100.0
4,caplcst,100.0
5,chldhm,100.0
6,dclvlf,100.0
7,dfincac,100.0
8,emplnof,100.0
9,emplnom,100.0


,cntry,n
0,IT,2865
1,GR,2757
2,UA,2661
3,DE,2420
4,AT,2354
5,BG,2239
6,HU,2118
7,IE,2017
8,ES,1844
9,FR,1771


In [6]:
'''with duckdb.connect(str(DB_PATH), read_only=True) as con:
    # 1. Check pruned columns with 100% NULL vs partial NULLs (35% - 99.9%)
    display(con.execute("""
        SELECT
            CASE WHEN null_pct = 100.0 THEN '100% NULL (inactive in ESS11)'
                 ELSE '35% - 99.9% NULL (sparse/split-ballot)' END AS category,
            COUNT(*) AS n_cols
        FROM l1_pruned_cols
        GROUP BY 1
    """).df())

    # 2. Verify that coalesced canonical variables survived in l1_ess11_clean
    kept = {c[0] for c in con.execute("DESCRIBE l1_ess11_clean").fetchall()}
    coalesced_rules = con.execute("""
        SELECT var_name, coalesce_from_json
        FROM l0_valid_cols
        WHERE coalesce_from_json != '[]'
    """).df()
    coalesced_rules["survived_in_l1"] = coalesced_rules["var_name"].isin(kept)
    display(coalesced_rules)'''

,category,n_cols
0,35% - 99.9% NULL (sparse/split-ballot),6
1,100% NULL (inactive in ESS11),205


,var_name,coalesce_from_json,survived_in_l1
0,dvrcdev,"[""dvrcdev"", ""dvrcdeva""]",True
1,edulvl,"[""edulvlb"", ""edulvla""]",True
2,edulvlf,"[""edulvlfb"", ""edulvlfa""]",True
3,edulvlm,"[""edulvlmb"", ""edulvlma""]",True
4,edulvlp,"[""edulvlpb"", ""edulvlpa""]",True
5,hinctnt,"[""hinctnta"", ""hinctnt""]",True
6,lvghw,"[""lvghwa"", ""lvghw""]",False
7,occf14,"[""occf14b"", ""occf14a"", ""occf14""]",True
8,pfmfdjb,"[""pfmfdjba"", ""pfmfdjb""]",False
9,rship10,"[""rshipa10"", ""rship10""]",False


In [7]:
for tmp_file in [TEMP_DIR / "ess11_raw_slice.parquet", TEMP_DIR / "l1_checkpoint.json"]:
    tmp_file.unlink(missing_ok=True)
print("[OK] L1 verified and temporary bridge files cleaned. Ready for S2.")

[OK] L1 verified and temporary bridge files cleaned. Ready for S2.
